In [ ]:
# --- CÉLULA ÚNICA: COLETA IBGE NO COLAB ---

# 1. Instalação Silenciosa das Bibliotecas
%pip install -q sidrapy pandas

import sys, subprocess, os

try:
    import sidrapy
except ModuleNotFoundError:
    # Install into the current Python environment and import again
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sidrapy", "-q"])
    import sidrapy
import pandas as pd

def get_dados_ibge():
    print("☁️ Conectando aos servidores do IBGE via Colab...")
    
    # --- A. Busca PIB per Capita (Tabela 5938) ---
    pib_data = sidrapy.get_table(
        table_code="5938", territorial_level="3", ibge_territorial_code="all",
        variable="37", period="last", header="n"
    )
    df_pib = pib_data.iloc[1:][['D1N', 'V']].copy()
    df_pib.columns = ['UF', 'PIB_per_capita']
    
    # --- B. Busca População (Tabela 4714 - Censo) ---
    pop_data = sidrapy.get_table(
        table_code="4714", territorial_level="3", ibge_territorial_code="all",
        variable="93", period="last", header="n"
    )
    df_pop = pop_data.iloc[1:][['D1N', 'V']].copy()
    df_pop.columns = ['UF', 'Populacao_2022']
    
    # --- C. Fusão e Limpeza ---
    print("⚙️ Processando dados na nuvem...")
    df_pib['PIB_per_capita'] = pd.to_numeric(df_pib['PIB_per_capita'])
    df_pop['Populacao_2022'] = pd.to_numeric(df_pop['Populacao_2022'])
    
    df_final = pd.merge(df_pib, df_pop, on='UF', how='inner')
    return df_final.sort_values('UF')

# --- EXECUÇÃO ---
try:
    df_resultado = get_dados_ibge()
    
    # Mostra uma amostra na tela
    print("\n✅ Dados Coletados com Sucesso:")
    print(df_resultado.head())
    
    # 1. Garante que a pasta 'dados' existe
    pasta_destino = 'dados'
    if not os.path.exists(pasta_destino):
        os.makedirs(pasta_destino)
    
    # 2. Define o caminho completo
    # Isso cria: C:\Users\Barth\projects_pc\cognicao\dados\1_ibge_socioeconomico.csv
    nome_arquivo = '1_ibge_socioeconomico.csv'
    caminho_completo = os.path.join(pasta_destino, nome_arquivo)
    
    # 3. Salva
    df_resultado.to_csv(caminho_completo, index=False)
    
    print(f"\n💾 Arquivo salvo em: {caminho_completo}")

except Exception as e:
    print(f"❌ Erro: {e}")

☁️ Conectando aos servidores do IBGE via Colab...
⚙️ Processando dados na nuvem...

✅ Dados Coletados com Sucesso:


,UF,PIB_per_capita,Populacao_2022
0,Acre,21374440,830018
12,Alagoas,76265620,3127683
4,Amapá,20099851,733759
1,Amazonas,131531038,3941613
14,Bahia,352617852,14141626



💾 Arquivo 1_ibge_socioeconomico.csv salvo no diretório atual.
